In [15]:
import pandas as pd
import os
from torch import device,cuda,float32,tensor,save
from sklearn.model_selection import train_test_split
from warnings import filterwarnings
filterwarnings("ignore")

%matplotlib inline

dev = device("cuda" if cuda.is_available() else "cpu")
dev

device(type='cuda')

In [16]:
df = pd.read_csv(r"D:\Data\Data_Entry_2017.csv")
train_val_names = pd.read_csv(r"D:\Data\train_val_list.txt",header=None,names=["Image Index"])
test_names = pd.read_csv(r"D:\Data\test_list.txt",header=None,names=["Image Index"])

In [17]:
PATH = r"D:\Data"
IMG_FOLDERS = [os.path.join(PATH,x+"\\images") for x in os.listdir(PATH) if x.startswith("images_")] 

NUM_OF_CLASSES = 14

ALL_DISEASES = sorted(df[df["Finding Labels"] != "No Finding"]["Finding Labels"].str.split("|").explode().unique())
ALL_DISEASES

['Atelectasis',
 'Cardiomegaly',
 'Consolidation',
 'Edema',
 'Effusion',
 'Emphysema',
 'Fibrosis',
 'Hernia',
 'Infiltration',
 'Mass',
 'Nodule',
 'Pleural_Thickening',
 'Pneumonia',
 'Pneumothorax']

In [18]:
train_val_names

,Image Index
0,00000001_000.png
1,00000001_001.png
2,00000001_002.png
3,00000002_000.png
4,00000004_000.png
...,...
86519,00030789_000.png
86520,00030793_000.png
86521,00030795_000.png
86522,00030801_000.png


In [19]:
test_names

,Image Index
0,00000003_000.png
1,00000003_001.png
2,00000003_002.png
3,00000003_003.png
4,00000003_004.png
...,...
25591,00030800_000.png
25592,00030802_000.png
25593,00030803_000.png
25594,00030804_000.png


In [20]:
train_val_names = set(train_val_names['Image Index'].values)
test_names = set(test_names['Image Index'].values)

In [21]:
df_imgs_labels = df[["Image Index","Finding Labels"]]
df_imgs_labels

,Image Index,Finding Labels
0,00000001_000.png,Cardiomegaly
1,00000001_001.png,Cardiomegaly|Emphysema
2,00000001_002.png,Cardiomegaly|Effusion
3,00000002_000.png,No Finding
4,00000003_000.png,Hernia
...,...,...
112115,00030801_001.png,Mass|Pneumonia
112116,00030802_000.png,No Finding
112117,00030803_000.png,No Finding
112118,00030804_000.png,No Finding


In [22]:
train_val_df = df_imgs_labels[df_imgs_labels['Image Index'].isin(train_val_names)]
train_val_df

,Image Index,Finding Labels
0,00000001_000.png,Cardiomegaly
1,00000001_001.png,Cardiomegaly|Emphysema
2,00000001_002.png,Cardiomegaly|Effusion
3,00000002_000.png,No Finding
12,00000004_000.png,Mass|Nodule
...,...,...
112100,00030789_000.png,Infiltration
112106,00030793_000.png,Mass|Nodule
112108,00030795_000.png,Pleural_Thickening
112114,00030801_000.png,No Finding


In [23]:
train_df,val_df = train_test_split(train_val_df,test_size=0.2,random_state=44)
test_df = df_imgs_labels[df_imgs_labels['Image Index'].isin(test_names)]

print("Train Shape : ", train_df.shape)
print("Validation Shape : ", val_df.shape)
print("Test Shape : ", test_df.shape)

train_df.shape[0] + val_df.shape[0] + test_df.shape[0]

Train Shape :  (69219, 2)
Validation Shape :  (17305, 2)
Test Shape :  (25596, 2)


112120

In [24]:
total = len(df)
pos_weights = []
for disease in ALL_DISEASES:
    pos = df["Finding Labels"].str.contains(disease).sum()
    neg = total - pos
    pos_weights.append(neg/pos)

pos_weights = tensor(pos_weights,dtype=float32).to(dev)
pos_weights

tensor([  8.6998,  39.3890,  23.0240,  47.6843,   7.4193,  43.5628,  65.5006,
        492.9207,   4.6359,  18.3912,  16.7097,  32.1226,  77.3508,  20.1467],
       device='cuda:0')

In [25]:
save({
    'train_df': train_df,
    'val_df': val_df,
    'test_df': test_df,
    'all_diseases': ALL_DISEASES,
    'img_folders': IMG_FOLDERS,
    "num_of_classes": NUM_OF_CLASSES,
    "pos_weights": pos_weights,
}, 'data_essentials.pth')